In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("SCFP2022.csv")

# Rebuild clean dataset
key_vars = ['YY1','Y1','WGT','AGE','AGECL','HHSEX','MARRIED','KIDS','EDUC','EDCL',
            'RACECL4','LF','INCOME','WAGEINC','NORMINC','IRAKH','THRIFT','RETQLIQ',
            'HRETQLIQ','ANYPEN','DCPLANCJ','DBPLANCJ','EDN_INST','HEDN_INST',
            'DEBT','NETWORTH','ASSET','FIN','CCBAL','NWCAT','INCCAT']
analysis = df[key_vars].copy()
continuous = ['WGT','INCOME','WAGEINC','NORMINC','IRAKH','THRIFT','RETQLIQ',
              'EDN_INST','DEBT','NETWORTH','ASSET','FIN','CCBAL','DCPLANCJ']
categorical = [c for c in key_vars if c not in continuous + ['YY1','Y1']]
avg_cont = analysis.groupby('YY1')[continuous].mean()
cat_vals = analysis.groupby('YY1')[categorical].first()
clean = pd.concat([avg_cont, cat_vals], axis=1).reset_index()

clean['has_student_debt'] = (clean['EDN_INST'] > 0).astype(int)
clean['has_ira'] = (clean['IRAKH'] > 0).astype(int)
clean['has_retirement'] = clean['HRETQLIQ'].astype(int)
age_labels = {1:'18-34',2:'35-44',3:'45-54',4:'55-64',5:'65-74',6:'75+'}
clean['age_group'] = clean['AGECL'].map(age_labels)
educ_labels = {1:'No HS diploma',2:'HS diploma/GED',3:'Some college',4:'College degree+'}
clean['educ_group'] = clean['EDCL'].map(educ_labels)
race_labels = {1:'White non-Hispanic',2:'Black/African-American',3:'Hispanic',4:'Other'}
clean['race_group'] = clean['RACECL4'].map(race_labels)
inc_labels = {1:'Bottom 20%',2:'20-39.9%',3:'40-59.9%',4:'60-79.9%',5:'80-89.9%',6:'Top 10%'}
clean['income_group'] = clean['INCCAT'].map(inc_labels)
nw_labels = {1:'Bottom 25%',2:'25-49.9%',3:'50-74.9%',4:'75-89.9%',5:'Top 10%'}
clean['nw_group'] = clean['NWCAT'].map(nw_labels)
clean['is_married'] = (clean['MARRIED'] == 1).astype(int)
clean['is_female'] = (clean['HHSEX'] == 2).astype(int)
clean['in_labor_force'] = clean['LF'].astype(int)
clean['debt_to_income'] = np.where(clean['INCOME'] > 0, clean['EDN_INST'] / clean['INCOME'], np.nan)

# ── 1. WEIGHTED COMPARISON: IRA ownership & balance by student debt status ────
def wavg(vals, wts):
    mask = (~np.isnan(vals)) & (~np.isnan(wts))
    return np.average(vals[mask], weights=wts[mask])

def wmedian(vals, wts, q=0.5):
    df2 = pd.DataFrame({'v': vals, 'w': wts}).dropna()
    df2 = df2.sort_values('v')
    cumw = df2['w'].cumsum()
    cutoff = df2['w'].sum() * q
    return float(df2.loc[cumw >= cutoff, 'v'].iloc[0])

print("=== 1. WEIGHTED SUMMARY: IRA by student debt ===")
for grp, label in [(0,'No student debt'),(1,'Has student debt')]:
    sub = clean[clean['has_student_debt'] == grp]
    w = sub['WGT']
    has_ira_rate = np.average(sub['has_ira'], weights=w)
    mean_ira = wavg(sub['IRAKH'].values, w.values)
    med_ira_all = wmedian(sub['IRAKH'].values, w.values)
    # median among those who have an IRA
    sub_ira = sub[sub['IRAKH'] > 0]
    med_ira_pos = wmedian(sub_ira['IRAKH'].values, sub_ira['WGT'].values)
    mean_ret = wavg(sub['RETQLIQ'].values, w.values)
    print(f"\n{label} (n={len(sub):,})")
    print(f"  % with IRA:              {has_ira_rate*100:.1f}%")
    print(f"  Mean IRA balance:        ${mean_ira:,.0f}")
    print(f"  Median IRA (all HHs):    ${med_ira_all:,.0f}")
    print(f"  Median IRA (IRA owners): ${med_ira_pos:,.0f}")
    print(f"  Mean retirement (total): ${mean_ret:,.0f}")

# ── 2. BY AGE GROUP ────────────────────────────────────────────────────────
print("\n\n=== 2. IRA OWNERSHIP RATE BY AGE x STUDENT DEBT ===")
age_order = ['18-34','35-44','45-54','55-64','65-74','75+']
results_age = []
for age in age_order:
    sub = clean[clean['age_group'] == age]
    for debt, dlabel in [(0,'No debt'),(1,'Has debt')]:
        s = sub[sub['has_student_debt'] == debt]
        if len(s) < 10:
            continue
        rate = np.average(s['has_ira'], weights=s['WGT'])
        mean_bal = wavg(s['IRAKH'].values, s['WGT'].values)
        results_age.append({'age': age, 'debt': dlabel, 'ira_rate': round(rate*100,1), 'mean_ira': round(mean_bal)})

age_df = pd.DataFrame(results_age)
print(age_df.to_string(index=False))

# ── 3. BY INCOME GROUP ─────────────────────────────────────────────────────
print("\n\n=== 3. IRA OWNERSHIP RATE BY INCOME x STUDENT DEBT ===")
inc_order = ['Bottom 20%','20-39.9%','40-59.9%','60-79.9%','80-89.9%','Top 10%']
results_inc = []
for inc in inc_order:
    sub = clean[clean['income_group'] == inc]
    for debt, dlabel in [(0,'No debt'),(1,'Has debt')]:
        s = sub[sub['has_student_debt'] == debt]
        if len(s) < 5:
            continue
        rate = np.average(s['has_ira'], weights=s['WGT'])
        mean_bal = wavg(s['IRAKH'].values, s['WGT'].values)
        results_inc.append({'income': inc, 'debt': dlabel, 'ira_rate': round(rate*100,1), 'mean_ira': round(mean_bal)})

inc_df = pd.DataFrame(results_inc)
print(inc_df.to_string(index=False))

# ── 4. LOGISTIC REGRESSION: IRA ownership ─────────────────────────────────
print("\n\n=== 4. LOGISTIC REGRESSION: P(has_ira) ===")
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

reg = clean.dropna(subset=['IRAKH','EDN_INST','INCOME','AGE','EDCL']).copy()
reg = reg[reg['INCOME'] > 0]

reg['log_income'] = np.log(reg['INCOME'])
features = ['has_student_debt','log_income','AGE','EDCL']
X = reg[features].values
y = reg['has_ira'].values
w = reg['WGT'].values

scaler = StandardScaler()
X_sc = scaler.fit_transform(X)

lr = LogisticRegression(max_iter=1000)
lr.fit(X_sc, y, sample_weight=w)

print("Feature coefficients (scaled):")
for feat, coef in zip(features, lr.coef_[0]):
    print(f"  {feat:<20} {coef:+.3f}")

# Unscaled odds ratio interpretation for has_student_debt
# Raw coefficient for student debt
idx = features.index('has_student_debt')
raw_coef = lr.coef_[0][idx] / scaler.scale_[idx]
odds_ratio = np.exp(raw_coef)
print(f"\nOdds ratio for has_student_debt: {odds_ratio:.3f}")
print(f"  → Having student debt is assoc. with {(1-odds_ratio)*100:.1f}% lower odds of having IRA (controlling for income, age, educ)")

# ── 5. REGRESSION: IRA balance (among IRA owners) ─────────────────────────
print("\n\n=== 5. OLS: Log IRA balance (IRA owners only) ===")
from sklearn.linear_model import LinearRegression

ira_owners = reg[reg['IRAKH'] > 0].copy()
ira_owners['log_ira'] = np.log(ira_owners['IRAKH'])
ira_owners['log_edn'] = np.log(ira_owners['EDN_INST'] + 1)

features2 = ['log_edn','log_income','AGE','EDCL']
X2 = ira_owners[features2].values
y2 = ira_owners['log_ira'].values
w2 = ira_owners['WGT'].values

sc2 = StandardScaler()
X2_sc = sc2.fit_transform(X2)
ols = LinearRegression()
ols.fit(X2_sc, y2, sample_weight=w2)
r2 = ols.score(X2_sc, y2)

print(f"R² = {r2:.3f}")
print("Coefficients (scaled):")
for f, c in zip(features2, ols.coef_):
    print(f"  {f:<20} {c:+.3f}")

# ── 6. Summary numbers for dashboard ──────────────────────────────────────
print("\n\n=== DASHBOARD NUMBERS ===")
# Weighted IRA rate gap by age
young = clean[clean['AGECL'].isin([1,2])]
for debt, label in [(0,'No debt'),(1,'Has debt')]:
    s = young[young['has_student_debt']==debt]
    r = np.average(s['has_ira'], weights=s['WGT'])
    print(f"Young HHs (18-44), {label}: IRA rate = {r*100:.1f}%")

# Median student debt amount
debtors = clean[clean['has_student_debt']==1]
med_debt = wmedian(debtors['EDN_INST'].values, debtors['WGT'].values)
print(f"\nMedian student debt (debtors): ${med_debt:,.0f}")
print(f"Mean student debt (debtors):  ${wavg(debtors['EDN_INST'].values, debtors['WGT'].values):,.0f}")



=== 1. WEIGHTED SUMMARY: IRA by student debt ===

No student debt (n=3,783)
  % with IRA:              32.2%
  Mean IRA balance:        $115,869
  Median IRA (all HHs):    $0
  Median IRA (IRA owners): $109,800
  Mean retirement (total): $212,478

Has student debt (n=812)
  % with IRA:              27.1%
  Mean IRA balance:        $23,719
  Median IRA (all HHs):    $0
  Median IRA (IRA owners): $22,000
  Mean retirement (total): $72,840


=== 2. IRA OWNERSHIP RATE BY AGE x STUDENT DEBT ===
  age     debt  ira_rate  mean_ira
18-34  No debt      19.7      8687
18-34 Has debt      25.1      7578
35-44  No debt      28.8     35700
35-44 Has debt      26.5     17500
45-54  No debt      27.7     79600
45-54 Has debt      28.8     31180
55-64  No debt      32.4    145236
55-64 Has debt      33.1     73365
65-74  No debt      43.9    219562
65-74 Has debt      15.0     10857
  75+  No debt      37.4    164071


=== 3. IRA OWNERSHIP RATE BY INCOME x STUDENT DEBT ===
    income     debt  ira_rat